In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from gensim.models import Word2Vec

# 1. Cargar datos
df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")
y = df['t']

# IMPORTANTE: Pon aquí la columna que ganó tu torneo de preprocesamiento
X_texto = df['text_stem_nltk']

# 2. Word2Vec necesita listas de palabras, no frases enteras. 
# Convertimos cada informe en una lista de tokens.
X_tokens = X_texto.apply(lambda x: str(x).split())

# 3. Split (Hacemos el split ANTES de entrenar Word2Vec para no hacer trampa)
X_train_tokens, X_test_tokens, y_train, y_test = train_test_split(
    X_tokens, y, test_size=0.20, random_state=42, stratify=y
)

print("🧠 Entrenando modelo Word2Vec médico desde cero...")

# 4. Entrenar Word2Vec solo con el Train
# vector_size=100 (cada palabra será un vector de 100 dimensiones)
# window=5 (mira 5 palabras a la izquierda y 5 a la derecha para entender el contexto)
# min_count=2 (ignora palabras que salen menos de 2 veces)
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=2, workers=4)

# 5. Función para convertir un informe entero en un solo vector
# (Hacemos la media de los vectores de todas las palabras de ese informe)
def document_vector(doc_tokens, model):
    # Filtramos palabras que el modelo conoce
    palabras_conocidas = [word for word in doc_tokens if word in model.wv.key_to_index]
    if len(palabras_conocidas) == 0:
        return np.zeros(model.vector_size)
    # Calculamos la media de los vectores de esas palabras
    return np.mean(model.wv[palabras_conocidas], axis=0)

print("🔄 Convirtiendo textos a vectores promediados...")

# 6. Transformar Train y Test a vectores numéricos
X_train_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_train_tokens])
X_test_w2v = np.array([document_vector(tokens, w2v_model) for tokens in X_test_tokens])

# 7. Entrenar y evaluar con un modelo clásico (Regresión Logística para comparar)
print("🚀 Entrenando clasificador sobre Embeddings...")
modelo_lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
modelo_lr.fit(X_train_w2v, y_train)

score_w2v = f1_score(y_test, modelo_lr.predict(X_test_w2v), average='macro')

print("-" * 50)
print(f"📊 Macro-F1 con Word2Vec (Propios) : {score_w2v:.4f}")
print("-" * 50)

🧠 Entrenando modelo Word2Vec médico desde cero...
🔄 Convirtiendo textos a vectores promediados...
🚀 Entrenando clasificador sobre Embeddings...
--------------------------------------------------
📊 Macro-F1 con Word2Vec (Propios) : 0.4908
--------------------------------------------------


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.layers import LSTM


X = df['text_stem_nltk']

mapeo_etiquetas = {'T1': 0, 'T2': 1, 'T3': 2, 'T4': 3}
y = df['t'].map(mapeo_etiquetas)

# 2. División Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Preparar el texto para la Red Neuronal (Tokenización y Padding)
vocab_size = 5000
max_length = 300 
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

pesos_clases = compute_class_weight(
    class_weight='balanced', classes=np.unique(y_train), y=y_train
)
dict_pesos = dict(enumerate(pesos_clases))


# 5. Construir la Arquitectura de la CNN
modelo_cnn = Sequential([
    # Capa Embedding: Aprende relaciones semánticas desde cero
    Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length),
    
    # Capa Convolucional: Lee ventanas de 3 palabras buscando patrones médicos
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    
    # Capa Max Pooling: Se queda con el patrón más fuerte (ej. el número del tamaño)
    GlobalMaxPooling1D(),
    
    # Capa Densa 
    Dense(64, activation='relu'),
    Dropout(0.5), # Apagamos neuronas aleatoriamente para evitar sobreajuste
    Dense(4, activation='softmax') # 4 salidas porque tenemos 4 estadios (T1, T2, T3, T4)
])

modelo_cnn.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)

# 6. Entrenamiento
historia = modelo_cnn.fit(
    X_train_pad, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.1, 
    class_weight=dict_pesos,
    verbose=2
)

modelo_lstm = Sequential([
    # Capa Embedding (Igual que en la CNN)
    Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length),
    
    # Capa LSTM: Lee de forma secuencial. 
    # Usamos 64 unidades de memoria y un dropout para no memorizar en exceso.
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    
    # Capa Densa Final (Clasificador)
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax') # 4 estadios (T1, T2, T3, T4)
])

modelo_lstm.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)

# 2. Entrenamiento de la LSTM
historia_lstm = modelo_lstm.fit(
    X_train_pad, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.1, 
    class_weight=dict_pesos,
    verbose=2
)

# 3. Evaluación
y_pred_prob_lstm = modelo_lstm.predict(X_test_pad)
y_pred_clases_lstm = np.argmax(y_pred_prob_lstm, axis=1)

score_lstm = f1_score(y_test, y_pred_clases_lstm, average='macro')

# 7. Evaluación con la métrica oficial (Macro-F1)
y_pred_prob = modelo_cnn.predict(X_test_pad)
y_pred_clases = np.argmax(y_pred_prob, axis=1)

score_cnn = f1_score(y_test, y_pred_clases, average='macro')

print("-" * 50)
print(f"📊 Macro-F1 con CNN  : {score_cnn:.4f} (del experimento anterior)")
print(f"📊 Macro-F1 con LSTM : {score_lstm:.4f}")
print("-" * 50)


Epoch 1/10


c:\Users\rv710\3_IA\2_Cuatrimestre\Procesamiento Lenguaje Natural I\Cancer-staging-proyect-pln\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


117/117 - 2s - 17ms/step - accuracy: 0.3636 - loss: 1.3358 - val_accuracy: 0.5133 - val_loss: 1.2190
Epoch 2/10
117/117 - 1s - 9ms/step - accuracy: 0.5034 - loss: 1.1213 - val_accuracy: 0.5617 - val_loss: 1.0428
Epoch 3/10
117/117 - 1s - 9ms/step - accuracy: 0.6011 - loss: 0.9300 - val_accuracy: 0.5956 - val_loss: 0.9787
Epoch 4/10
117/117 - 1s - 9ms/step - accuracy: 0.6844 - loss: 0.7361 - val_accuracy: 0.5884 - val_loss: 0.9685
Epoch 5/10
117/117 - 1s - 9ms/step - accuracy: 0.7482 - loss: 0.5922 - val_accuracy: 0.6174 - val_loss: 0.8905
Epoch 6/10
117/117 - 1s - 9ms/step - accuracy: 0.8311 - loss: 0.4338 - val_accuracy: 0.6029 - val_loss: 0.9261
Epoch 7/10
117/117 - 1s - 9ms/step - accuracy: 0.8861 - loss: 0.3085 - val_accuracy: 0.6271 - val_loss: 0.9580
Epoch 8/10
117/117 - 1s - 9ms/step - accuracy: 0.9278 - loss: 0.2163 - val_accuracy: 0.6126 - val_loss: 1.0838
Epoch 9/10
117/117 - 1s - 9ms/step - accuracy: 0.9407 - loss: 0.1930 - val_accuracy: 0.6247 - val_loss: 1.0566
Epoch 10/10

In [ ]:
from tensorflow.keras.layers import LSTM
modelo_lstm = Sequential([
    # Capa Embedding (Igual que en la CNN)
    Embedding(input_dim=vocab_size, output_dim=100, input_length=max_length),
    
    # Capa LSTM: Lee de forma secuencial. 
    # Usamos 64 unidades de memoria y un dropout para no memorizar en exceso.
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    
    # Capa Densa Final (Clasificador)
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax') # 4 estadios (T1, T2, T3, T4)
])

modelo_lstm.compile(
    loss='sparse_categorical_crossentropy', 
    optimizer='adam', 
    metrics=['accuracy']
)

# 2. Entrenamiento de la LSTM
historia_lstm = modelo_lstm.fit(
    X_train_pad, y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.1, 
    class_weight=dict_pesos,
    verbose=2
)

# 3. Evaluación
y_pred_prob_lstm = modelo_lstm.predict(X_test_pad)
y_pred_clases_lstm = np.argmax(y_pred_prob_lstm, axis=1)

score_lstm = f1_score(y_test, y_pred_clases_lstm, average='macro')

print("-" * 50)
print(f"📊 Macro-F1 con CNN  : {score_cnn:.4f} ")
print(f"📊 Macro-F1 con LSTM : {score_lstm:.4f}")
print("-" * 50)

Epoch 1/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 13s 95ms/step - accuracy: 0.2699 - loss: 1.3893 - val_accuracy: 0.2470 - val_loss: 1.3830
Epoch 2/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 14s 119ms/step - accuracy: 0.3035 - loss: 1.3734 - val_accuracy: 0.2252 - val_loss: 1.3608
Epoch 3/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 21s 120ms/step - accuracy: 0.3377 - loss: 1.3094 - val_accuracy: 0.2373 - val_loss: 1.3629
Epoch 4/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 18s 95ms/step - accuracy: 0.3625 - loss: 1.2257 - val_accuracy: 0.3462 - val_loss: 1.3748
Epoch 5/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 90ms/step - accuracy: 0.3997 - loss: 1.1581 - val_accuracy: 0.2131 - val_loss: 1.4183
Epoch 6/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 92ms/step - accuracy: 0.4223 - loss: 1.0705 - val_accuracy: 0.2494 - val_loss: 1.4893
Epoch 7/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 92ms/step - accuracy: 0.4500 - loss: 1.0202 - val_accuracy: 0.2518 - val_loss: 1.5785
Epoch 8/10
117/117 ━━━━━━━━━━━━━━━━━━━━ 11s 91ms/step - accuracy: 0.4794 - loss: 0.9715 